In [1]:
#   numpy(넘파이): 숫자 여러 개를 배열로 묶어 한 번에 계산하게 해 주는 라이브러리입니다.
#                 데이터 준비와 정규화(NumPy 흐름)에 사용합니다.
import numpy as np

#   tensorflow: 이번 노트북의 주인공입니다.
#       - tf.keras.Sequential           : 모델을 쌓아 만드는 고수준 API
#       - tf.keras.layer.Dense          : H = a·X_norm + b를 계산하는 선형 부품 (PyTorch의 Linear에 대응)
#       - tf.keras.losses.BinaryCrossentropy(from_logits=True): H를 받아 sigmoid + BCE를 내부 처리  (BCEWithLogitsLoss에 대응)
#       - tf.keras.optimizers.SGD       : 파라미터 업데이트 (PyTorch의 SGD optimizer에 대응)
import tensorflow as tf

#   pandas(판다스): 계산 결과를 '표(DataFrame)' 형태로 보기 좋게 정리해주는 라이브러리입니다.
#       - 바로 아래 '1_ sigmoid 복습' 준비 실습에서 H와 z(=sigmoid(H))를 표로 나란히 비교할 때만 사용합니다.
#       - (모델 학습에 꼭 필요한 것은 아니고, 눈으로 확인하기 쉽게 하려는 용도입니다.)
import pandas as pd

In [2]:
#   seed 고정: 실행할 때마다 초기 weight가 비슷해져 결과를 비교하기 쉽습니다.
#       - np.random.seed(42)    : NumPy 쪽 난수 고정
#       - tf.random.set.seed(42): TensorFlow 쪽 난수 고정(Dense layer 초기 weight에 영향)
np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow 버전: ', tf.__version__)
print('NumPy 버전', np.__version__)
print('Pandas 버전', pd.__version__)

TensorFlow 버전:  2.10.0
NumPy 버전 1.23.5
Pandas 버전 2.3.3


In [3]:
#   H는 sigmoid 이전의 선형 출력값입니다.
#   H는 확률이 아니므로 음수, 0, 양수, 1보다 큰 값이 모두 가능합니다.
#   H는 선형 계산 결과이므로 -3, -1, 0, 1, 3처럼 다양한 실수값이 될 수 있습니다.
#   (앞 셀에서 import한 numpy(np), tensorflow(tf), pandas(pd)를 그대로 사용합니다.)
#
#   여기서는 H 값을 직접 만든 뒤 sigmoid를 적용해 보기 위해 NumPy 배열을 사용합니다.
#   아직 모델을 학습한느 단계가 아니므로 TensorFlow Tensor로 만들 필요는 없습니다.
#
#   shape을 (5,1 )로 만든 이유:
#       - 데이터가 5개 있습니다.
#       - 각 데이터는 H 값 1개를 가집니다.
#       - 따라서 모양은 (데이터 개수, 1) = (5, 1)입니다.
H_example = np.array([[-3.0], [-1.0], [0.0], [1.0], [3.0]])

H_example

array([[-3.],
       [-1.],
       [ 0.],
       [ 1.],
       [ 3.]])

In [4]:
#   z는 sigmoid(H)로 얻는 값입니다.
#   z는 항상 0과 1 사이의 값이므로 예측 확률로 해석할 수 있습니다.
#
#   [NumPy 배열 → tf.sigmoid → TensorFlow Tensor]
#       H_example은 Numpy 배열입니다.
#       tf.sigmoid()는 NumPy 배열도 입력으로 받을 수 있습니다.
#       tf.sigmoid(H_example)을 적용하면 TensorFlow가 이 값을 받아 계산하고,
#       그 결과인 z_example은 TensorFlow Tensor로 반환됩니다.
z_example = tf.sigmoid(H_example)

z_example

<tf.Tensor: shape=(5, 1), dtype=float64, numpy=
array([[0.04742587],
       [0.26894142],
       [0.5       ],
       [0.73105858],
       [0.95257413]])>

In [7]:
#   H와 z를 표로 나란히 놓고 비교합니다.    (pandas DataFrame 사용)
#
#   [pd.DataFrame 이란?]
#       pd.DataFrame은 값을 '표(table)' 형태로 보여 주기 위한 도구입니다.
#       모델 학습에 꼭 필요한 코드는 아니지만, H와 z를 나란히 비교하면
#       'H는 확률이 아니고, z가 확률이다'라는 점을 눈으로 확인하기 쉽습니다.
#
#   [reshape(-1) / .numpy() 가 왜 필요한가?]
#       H_example, z_example 은 shape이 (5, 1)인 2차원(5행 1열) 데이터입니다.
#       DataFrame의 한 '열(Columns)'로 넣으려면 [값1, 값2, 값3, ...] 형태의 1차원 배열이 편합니다.
#           - .reshape(-1)  : (5, 1) 같은 2차원 모양을 (5,) 1차원으로 '펴 줍니다'.  (-1은 길이를 알아서 맞추라는 뜻)
#           - .numpy()      : TensorFlow Tensor를 NumPy 배열로 바꿉니다.    (Tensor에만 필요)
sigmoid_result_df = pd.DataFrame({
    #   H_example은 NumPy 배열이므로, .numpy()가 필요 없습니다.
    #   reshape(-1)은 (5, 1) 모양을 DataFrame 열에 넣기 쉬운 1차원 형태로 펴는 역할입니다.
    'H': H_example.reshape(-1),
    
    #   z_example은 TensorFlow Tensor이므로 .numpy()로 NumPy 배열로 꺼냅니다.
    'z = sigmoid(H)': z_example.numpy().reshape(-1) # sigmoid를 통과한 예측 확률    (항상 0~1 사이)
})

#   sigmoid 함수는 H를 0과 1 사이의 값으로 바꿉니다.
#   따라서 sigmoid(H)의 결과인 z를 예측 확률로 해석합니다.
#       정리: H = sigmoid 이전의 선형 출력값
#            z = sigmoid(H) = 예측 확률
sigmoid_result_df

,H,z = sigmoid(H)
0,-3.0,0.047426
1,-1.0,0.268941
2,0.0,0.500000
3,1.0,0.731059
4,3.0,0.952574


In [8]:
#   이진 분류에서는 예측 확률 z를 기준으로 최종 클래스를 결정합니다.
#   이번 입문 예제에서는 기본 기준값으로 0.5를 사용합니다.
#       z ≥ 0.5 이면 1
#       z < 0.5 이면 0
#
#   [tf.case(z_example ≥ 0.5, tf.int32) 을 초보자용으로 풀어 보면]
#       1) z_example ≥ 0.5 → 각 원소가 조건을 만족하는지에 따라 True 또는 False 값을 만듭니다.
#       2) tf.case(..., tf.int32) → True를 1로, False를 0으로 '형 변환(case)'합니다.
#       즉, 확률 z를 최종 분류값 0 또는 1로 바꾸는 코드입니다.
prediction_example = tf.cast(z_example >= 0.5, tf.int32)

#   앞에서 만든 표(sigmoid_result_df)에 prediction 열을 추가해 H, z, 최종 분류를 한눈에 봅니다.
#   (여기서도 .numpy().reshape(-1)로 Tensor를 1차원 NumPy 배열로 펴서 한 열로 넣습니다.)
sigmoid_result_df['prediction'] = prediction_example.numpy().reshape(-1)

sigmoid_result_df

,H,z = sigmoid(H),prediction
0,-3.0,0.047426,0
1,-1.0,0.268941,0
2,0.0,0.500000,1
3,1.0,0.731059,1
4,3.0,0.952574,1


In [10]:
#   ==============================================================
#   3. Sequential 모델 생성 (Dense(1) = PyTorch의 Linear(1, 1))
#   ==============================================================

#   tf.keras.Input(shape=(1,)) : 입력이 '데이터 한 개당 특성 1개'라는 것을 알려 줍니다.
#                                (데이터 개수가 아니라 '특성 개수'라는 점에 주의!)
#   tf.keras.layers.Dense(1)   : H = a·X_norm + b를 계산하는 선형 층입니다.
#       ★ activation을 넣지 않습니다! 따라서 출력은 확률 z가 아니라 H(선형 계산값)입니다.
#          (sigmoid는 뒤에서 BinaryCrossentropy(from_logits=True)가 내부 처리합니다.)

#   모델을 만들기 직전에 seed를 한 번 더 곶어합니다.
#       주의: tf.constant나 tf.sigmoid는 난수를 사용하는 연산이 아닙니다.
#            따라서 위 '1) sigmoid 복습' 준비 실습 자체가 Dense 초기 weight를 바꾸는 원인은 아닙니다.
#       그렇다면 왜 다시 고정할까요?
#           - 교육용 노트북에서는 모델 생성 직전에 seed를 다시 고정해 두면
#              '이제부터 모델을 만들겠다'는 초기화 시점을 명확히 표시할 수 있고,
#               실행 결과를 항상 같은 초기 weight에서 비교하기 쉽습니다.    (재현성 표시)
tf.random.set_seed(42)

model = tf.keras.Sequential([
    tf.keras.Input(shape = (1,)),
    tf.keras.layers.Dense(1)
])

#   모델 구조를 요약해서 봅니다.
#       아래 출력에서 Param # 가 2로 보이는 이유는
#       Dense(1)이 weight a 1개 + bias b 1개 = 총 2개의 파라미터를 가지기 때문입니다.
#       즉, 이 모델의 학습으로 바꾼느 값은 a와 b 두 개 뿐입니다.
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 1)                 2         
                                                                 
Total params: 2
Trainable params: 2
Non-trainable params: 0
_________________________________________________________________
